## Hướng Dẫn Chuyển File Google Drive Giữa Các Tài Khoản (Nhanh & Dễ - 2025)

Hướng dẫn từng bước cách chuyển file Google Drive từ tài khoản này sang tài khoản khác bằng Google Colab! Tự động hóa quy trình mà không cần tải xuống hay tải lên thủ công. Phù hợp cho người mới, sinh viên và chuyên gia quản lý nhiều tài khoản Drive.

In [ ]:
#@title 🔐 Quản Lý Quyền Truy Cập Google Drive
#@markdown ## 1. Xác Thực & Liệt Kê Shared Drive
#@markdown Chạy cell này trước. Nó sẽ kết nối tài khoản Google của bạn và liệt kê tất cả Shared Drive bạn có quyền truy cập.
#@markdown Bạn có thể sao chép tên Shared Drive từ kết quả để sử dụng ở bước tiếp theo.

# --- Cài Đặt & Xác Thực ---
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib -q
from google.colab import auth
from googleapiclient.discovery import build
import google.auth
from googleapiclient.http import BatchHttpRequest
import time
from tqdm.notebook import tqdm
import os

# Xác thực người dùng
auth.authenticate_user()
# Lấy thông tin xác thực và tạo dịch vụ Drive
# Get default credentials and build the Drive service
creds, _ = google.auth.default()
drive_service = build('drive', 'v3', credentials=creds)

print("✅ Xác thực thành công.")
# --- Liệt kê Shared Drive cho tiện ---
# --- List Shared Drives for user convenience ---
try:
    print("\n--- Danh Sách Shared Drive ---")
    drives_response = drive_service.drives().list().execute()
    shared_drives = drives_response.get('drives', [])
    if not shared_drives:
        print("Không tìm thấy Shared Drive nào.")
    else:
        for drive in shared_drives:
            print(f"- Tên: '{drive.get('name')}' (ID: {drive.get('id')})")
    print("-----------------------------\n")
except Exception as e:
    print(f"⚠️ Không thể liệt kê Shared Drive: {e}")


#@markdown ---
#@markdown ## 2. Cấu Hình & Thực Thi
#@markdown Điền form bên dưới. Nếu muốn thao tác trên Shared Drive, dán tên từ danh sách phía trên. Để trống 'shared_drive_name' nếu dùng 'My Drive'.

#@markdown ### **Hành Động**
#@markdown Chọn thêm hoặc gỡ quyền truy cập.
action = "Share" #@param ["Share", "Unshare"]

#@markdown ### **Drive Nguồn (Tùy Chọn)**
#@markdown Nhập tên chính xác của Shared Drive. **Để trống nếu dùng 'My Drive'.**
shared_drive_name = "" #@param {type:"string"}

#@markdown ### **Đường Dẫn Nguồn**
#@markdown Nhập đường dẫn đầy đủ đến file hoặc thư mục, tương đối với drive đã chọn.
#@markdown *Ví dụ (thư mục): `/My Project Folder`*
#@markdown *Ví dụ (file): `/My Project Folder/document.docx`*
source_path = "" #@param {type:"string"}

#@markdown ### **Email Người Nhận**
#@markdown Nhập địa chỉ email của người dùng mà bạn muốn quản lý quyền.
destination_email = "" #@param {type:"string"}

#@markdown ### **Phạm Vi Chia Sẻ (chỉ cho thư mục)**
#@markdown Nếu đường dẫn là thư mục, chọn loại nội dung cần áp dụng.
sharing_scope = "Files and Folders" #@param ["Files and Folders", "Files Only", "Folders Only"]

#@markdown ### **Vai Trò Chia Sẻ (chỉ cho hành động 'Share')**
#@markdown Chọn mức quyền cấp cho người nhận.
sharing_role = "writer" #@param ["writer", "commenter", "reader"]

#@markdown ### **Thông Báo (chỉ cho hành động 'Share')**
#@markdown Tích ô này nếu muốn gửi thông báo email cho người nhận.
send_notification = False #@param {type:"boolean"}


def get_item_id_from_path(drive_id, is_shared_drive, path):
    """
    Translates a path into a Google Drive file/folder ID and its mimeType,
    supporting both My Drive and Shared Drives.
    """
    clean_path = path.strip().strip('/')
    path_components = clean_path.split('/') if clean_path else []

    current_id = drive_id
    item_info = None

    for component in path_components:
        if not component: continue
        query = f"name='{component}' and '{current_id}' in parents and trashed=false"
        try:
            list_request = drive_service.files().list(
                q=query, fields='files(id, name, mimeType)', pageSize=2,
                supportsAllDrives=True, includeItemsFromAllDrives=True)
            if is_shared_drive:
                list_request.driveId = drive_id
                list_request.corpora = 'drive'
            else:
                 list_request.corpora = 'user'
            response = list_request.execute()
            files = response.get('files', [])
            if not files:
                print(f"❌ LỖI: Không tìm thấy thành phần đường dẫn '{component}'.")
                return None, None
            if len(files) > 1: print(f"⚠️ Cảnh báo: Tìm thấy nhiều mục tên '{component}'. Sử dụng mục đầu tiên.")
            item_info = files[0]
            current_id = item_info['id']
        except Exception as e:
            print(f"❌ Lỗi khi phân giải thành phần đường dẫn '{component}': {e}")
            return None, None

    if not path_components:
        item_info = drive_service.files().get(fileId=drive_id, fields='id, name, mimeType', supportsAllDrives=True).execute()

    return item_info.get('id'), item_info.get('mimeType')

def manage_permissions_by_path():
    """
    Finds the item at the specified path and shares or unshares it (or its contents).
    """
    # --- Step 1: Determine the target drive ---
    target_drive_id = 'root'
    is_shared_drive = False
    drive_display_name = "'My Drive'"

    if shared_drive_name:
        print(f"Đang tìm Shared Drive: '{shared_drive_name}'...")
        try:
            drive_response = drive_service.drives().list(q=f"name = '{shared_drive_name}'").execute()
            drives = drive_response.get('drives', [])
            if not drives:
                print(f"❌ LỖI: Không tìm thấy Shared Drive '{shared_drive_name}'.")
                return
            target_drive_id = drives[0]['id']
            drive_display_name = f"Shared Drive '{shared_drive_name}'"
            is_shared_drive = True
            print(f"✅ Đã tìm thấy {drive_display_name} (ID: {target_drive_id})")
        except Exception as e:
            print(f"❌ Lỗi khi tìm Shared Drive: {e}")
            return

    # --- Step 2: Resolve the path to an item ID ---
    print(f"\nĐang phân giải đường dẫn: '{source_path}' trong {drive_display_name}...")
    item_id, mime_type = get_item_id_from_path(target_drive_id, is_shared_drive, source_path)
    if not item_id: return

    # --- Step 3: Get list of items to process based on path and scope ---
    items_to_process = []
    if mime_type == 'application/vnd.google-apps.folder':
        print(f"✅ Đường dẫn trỏ tới thư mục. Áp dụng phạm vi: '{sharing_scope}'.")
        base_query = f"'{item_id}' in parents and trashed=false"
        scope_query = ""
        if sharing_scope == "Files Only": scope_query = " and mimeType != 'application/vnd.google-apps.folder'"
        elif sharing_scope == "Folders Only": scope_query = " and mimeType = 'application/vnd.google-apps.folder'"
        list_request = drive_service.files().list(
            q=base_query + scope_query, fields="nextPageToken, files(id, name)", pageSize=1000,
            supportsAllDrives=True, includeItemsFromAllDrives=True)
        if is_shared_drive: list_request.driveId = target_drive_id; list_request.corpora = 'drive'
        else: list_request.corpora = 'user'
        page_token = None
        while True:
            try:
                if page_token: list_request.pageToken = page_token
                response = list_request.execute()
                items_to_process.extend(response.get('files', []))
                page_token = response.get('nextPageToken', None)
                if page_token is None: break
            except Exception as e:
                print(f"❌ Lỗi khi liệt kê nội dung thư mục: {e}")
                return
    else:
        print("✅ Đường dẫn trỏ tới một file đơn.")
        item_name_req = drive_service.files().get(fileId=item_id, fields='name', supportsAllDrives=True).execute()
        items_to_process.append({'id': item_id, 'name': item_name_req.get('name', 'Unknown File')})

    if not items_to_process:
        print("📁 Không tìm thấy mục nào để xử lý theo đường dẫn và phạm vi đã chọn.")
        return

    # --- Step 4: Execute the chosen action ---
    print(f"\nChuẩn bị {action.upper()} {len(items_to_process)} mục cho '{destination_email}'...")

    if action == "Share":
        permission = {'type': 'user', 'role': sharing_role, 'emailAddress': destination_email}
        batch = drive_service.new_batch_http_request()
        for item in items_to_process:
            batch.add(drive_service.permissions().create(
                fileId=item['id'], body=permission,
                sendNotificationEmail=send_notification,
                supportsAllDrives=True))
        print("Đang thực thi yêu cầu chia sẻ hàng loạt...")
        try:
            batch.execute()
        except Exception as e:
            print(f"❌ Chia sẻ hàng loạt thất bại: {e}")

    elif action == "Unshare":
        permissions_to_delete = []
        print("Đang tìm quyền hiện có để gỡ...")
        for item in tqdm(items_to_process, desc="Đang kiểm tra quyền"):
            try:
                perms = drive_service.permissions().list(fileId=item['id'], fields='permissions(id, emailAddress)', supportsAllDrives=True).execute()
                for p in perms.get('permissions', []):
                    if p.get('emailAddress', '').lower() == destination_email.lower():
                        permissions_to_delete.append({'fileId': item['id'], 'permissionId': p['id']})
                        break
            except Exception as e:
                print(f"\n⚠️ Không thể kiểm tra quyền cho '{item['name']}'. Bỏ qua. Lý do: {e}")

        if not permissions_to_delete:
            print(f"Không tìm thấy quyền nào của '{destination_email}' trên các mục đã chọn.")
            return

        print(f"\nTìm thấy {len(permissions_to_delete)} quyền cần gỡ. Đang thực thi xóa hàng loạt...")
        batch = drive_service.new_batch_http_request()
        for perm in permissions_to_delete:
            batch.add(drive_service.permissions().delete(fileId=perm['fileId'], permissionId=perm['permissionId'], supportsAllDrives=True))
        try:
            batch.execute()
        except Exception as e:
            print(f"❌ Gỡ chia sẻ hàng loạt thất bại: {e}")

    print(f"\n✅ Đã xử lý xong tất cả {len(items_to_process)} mục.")

# --- Execute the Main Function ---
if destination_email and "@" in destination_email:
    manage_permissions_by_path()
else:
    print("Vui lòng nhập địa chỉ email người nhận hợp lệ trước khi chạy.")


In [ ]:
#@title 📋 Sao Chép 'Được Chia Sẻ Với Tôi' Sang My Drive
#@markdown ## 1. Xác Thực & Thiết Lập
#@markdown Chạy cell này để cài đặt thư viện, kết nối tài khoản Google, và thiết lập dịch vụ Drive với timeout dài cho các thao tác lớn.

# --- Cài Đặt & Xác Thực ---
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib -q
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import BatchHttpRequest
import google.auth
from google.auth.transport.requests import Request
import requests
import time
import math
from tqdm.notebook import tqdm

# --- Xác Thực ---
print("Đang xác thực người dùng...")
auth.authenticate_user()

# --- Thiết Lập Dịch Vụ Drive Với Timeout Tăng ---
print("Đang thiết lập dịch vụ Google Drive...")
creds, _ = google.auth.default()
# Sử dụng session với timeout dài (600s = 10 phút) cho các thao tác mạng
session = requests.Session()
session.timeout = 600
transport = Request(session)
creds.refresh(transport)
# Tạo dịch vụ với thông tin xác thực chứa cấu hình timeout
drive_service = build('drive', 'v3', credentials=creds)
print("✅ Xác thực thành công. Bạn có thể cấu hình quá trình sao chép bên dưới.")

#@markdown ---
#@markdown ## 2. Cấu Hình & Chạy Quá Trình Sao Chép
#@markdown Điền form bên dưới, sau đó chạy cell này để bắt đầu sao chép.

#@markdown ### **Mục Nguồn (Tùy Chọn)**
#@markdown Để sao chép **một mục cụ thể**, nhập tên chính xác ở đây.
#@markdown **Để trống để xử lý TẤT CẢ mục** trong danh sách 'Được chia sẻ với tôi'.
specific_item_name = "" #@param {type:"string"}

#@markdown ### **Thư Mục Đích**
#@markdown Nhập tên thư mục trong 'My Drive' để sao chép vào.
#@markdown **Để trống để sao chép vào thư mục gốc 'My Drive'.**
destination_folder_name = "" #@param {type:"string"}

#@markdown ### **Phạm Vi Nội Dung**
#@markdown Nếu sao chép tất cả hoặc một thư mục cụ thể, chọn loại nội dung cần sao chép.
copy_scope = "Files and Folders" #@param ["Files and Folders", "Files Only", "Folders Only"]

#@markdown ### **Hành Vi**
#@markdown Tích ô này để bỏ qua file/thư mục đã tồn tại (theo tên) ở đích.
skip_existing_items = True #@param {type:"boolean"}

#@markdown ### **Cài Đặt Nâng Cao**
#@markdown Điều chỉnh cài đặt này để tối ưu hiệu suất cho các lần chuyển lớn hoặc chậm.
max_retries_per_chunk = 5 #@param {type:"integer"}
initial_retry_delay_seconds = 15 #@param {type:"integer"}


# --- Helper function to get existing items in a folder ---
def get_existing_items(folder_id):
    """Lấy tất cả tên file và thư mục trong thư mục cho việc tra cứu nhanh."""
    existing_items = {}
    page_token = None
    query = f"'{folder_id}' in parents and trashed = false"
    while True:
        try:
            response = drive_service.files().list(
                q=query,
                fields="nextPageToken, files(id, name, mimeType)",
                pageSize=1000,
                pageToken=page_token
            ).execute()
            for item in response.get('files', []):
                if item['name'] not in existing_items:
                    existing_items[item['name']] = {'id': item['id'], 'mimeType': item['mimeType']}
            page_token = response.get('nextPageToken', None)
            if page_token is None: break
        except Exception as e:
            print(f"  - Cảnh báo: Không thể liệt kê mục hiện có trong thư mục {folder_id}: {e}")
            break
    return existing_items

# --- Function to create batches in chunks with robust retries ---
def execute_batch_in_chunks(items, build_request_func):
    """Thực thi danh sách mục theo nhóm với logic thử lại và exponential backoff."""
    batch_size_limit = 999
    num_items = len(items)
    if num_items == 0: return

    num_batches = math.ceil(num_items / batch_size_limit)
    progress_bar = tqdm(total=num_items, desc="Đang xử lý hàng loạt")

    for i in range(num_batches):
        start_index = i * batch_size_limit
        end_index = start_index + batch_size_limit
        chunk = items[start_index:end_index]

        retry_delay = initial_retry_delay_seconds
        for attempt in range(max_retries_per_chunk):
            batch = drive_service.new_batch_http_request()
            for item in chunk:
                build_request_func(batch, item)

            try:
                if len(batch._requests) > 0:
                    batch.execute()
                progress_bar.update(len(chunk))
                break
            except Exception as e:
                print(f"      - Lần thử {attempt + 1}/{max_retries_per_chunk} thất bại: {e}")
                if attempt + 1 == max_retries_per_chunk:
                    print(f"      - Đã hết số lần thử. Bỏ qua nhóm {len(chunk)} mục này.")
                    progress_bar.update(len(chunk)) # Mark as processed to not hang the bar
                    break
                print(f"      - Thử lại sau {retry_delay} giây...")
                time.sleep(retry_delay)
                retry_delay *= 2
    progress_bar.close()

# --- Function to recursively copy folder contents ---
def copy_folder_contents(source_folder_id, destination_folder_id, source_folder_name):
    """Sao chép nội dung thư mục nguồn một cách đệ quy và hiệu quả."""
    print(f"\n📁 Đang xử lý thư mục con: {source_folder_name}")

    # List items in source folder
    source_files, source_folders = [], []
    page_token = None
    while True:
        try:
            response = drive_service.files().list(
                q=f"'{source_folder_id}' in parents and trashed = false",
                fields="nextPageToken, files(id, name, mimeType)", pageSize=1000, pageToken=page_token).execute()
            for item in response.get('files', []):
                (source_folders if item['mimeType'] == 'application/vnd.google-apps.folder' else source_files).append(item)
            page_token = response.get('nextPageToken', None)
            if page_token is None: break
        except Exception as e:
            print(f"  - Lỗi liệt kê nội dung nguồn '{source_folder_name}': {e}. Bỏ qua thư mục này.")
            return

    if copy_scope == "Folders Only":
        source_files = [] # Don't copy files if only folders are selected

    existing_dest_items = get_existing_items(destination_folder_id) if skip_existing_items else {}
    files_to_batch = [f for f in source_files if f['name'] not in existing_dest_items]
    folders_to_batch = [f for f in source_folders if f['name'] not in existing_dest_items]
    print(f"  - Tìm thấy {len(files_to_batch)} file mới và {len(folders_to_batch)} thư mục mới cần sao chép.")

    if files_to_batch:
        def build_file_copy_request(batch, file_item):
            batch.add(drive_service.files().copy(fileId=file_item['id'], body={'name': file_item['name'], 'parents': [destination_folder_id]}))
        execute_batch_in_chunks(files_to_batch, build_file_copy_request)

    newly_created_folders = {}
    if folders_to_batch:
        def folder_creation_callback(request_id, response, exception):
            if not exception: newly_created_folders[request_id] = response
        def build_folder_create_request(batch, folder_item):
            metadata = {'name': folder_item['name'], 'mimeType': 'application/vnd.google-apps.folder', 'parents': [destination_folder_id]}
            batch.add(drive_service.files().create(body=metadata, fields='id, name'), request_id=folder_item['id'], callback=folder_creation_callback)
        execute_batch_in_chunks(folders_to_batch, build_folder_create_request)

    for source_folder in source_folders:
        dest_folder_id_for_recursion = None
        if source_folder['id'] in newly_created_folders:
            dest_folder_id_for_recursion = newly_created_folders[source_folder['id']]['id']
        elif skip_existing_items and source_folder['name'] in existing_dest_items:
            dest_folder_id_for_recursion = existing_dest_items[source_folder['name']]['id']
        if dest_folder_id_for_recursion:
            copy_folder_contents(source_folder['id'], dest_folder_id_for_recursion, source_folder['name'])


# --- Main script execution ---
def main():
    print("--- Bắt Đầu Sao Chép Từ 'Được Chia Sẻ Với Tôi' ---")
    # Step 1: Determine the destination folder ID
    if destination_folder_name:
        print(f"Đang tìm thư mục đích: '{destination_folder_name}'...")
        try:
            response = drive_service.files().list(
                q=f"name='{destination_folder_name}' and mimeType='application/vnd.google-apps.folder' and 'root' in parents and trashed=false",
                fields='files(id, name)').execute()
            if not response['files']:
                print(f"❌ Không tìm thấy thư mục đích '{destination_folder_name}' trong 'My Drive'. Vui lòng tạo trước.")
                return
            dest_root_id = response['files'][0]['id']
            print(f"✅ Đã đặt đích là '{destination_folder_name}' (ID: {dest_root_id})")
        except Exception as e:
            print(f"❌ Lỗi tìm thư mục đích: {e}")
            return
    else:
        dest_root_id = drive_service.files().get(fileId='root', fields='id').execute()['id']
        print("✅ Đã đặt đích là thư mục gốc 'My Drive'.")

    # Step 2: Get the list of items to process
    items_to_process = []
    if specific_item_name:
        print(f"\nĐang tìm trong 'Được chia sẻ với tôi' mục tên: '{specific_item_name}'...")
        query = f"name = '{specific_item_name}' and sharedWithMe and trashed = false"
        try:
            response = drive_service.files().list(q=query, fields="files(id, name, mimeType)", pageSize=10).execute()
            found_items = response.get('files', [])
            if not found_items:
                print(f"❌ Không tìm thấy mục tên '{specific_item_name}' trong danh sách 'Được chia sẻ với tôi'.")
                return
            if len(found_items) > 1:
                print(f"⚠️ Cảnh báo: Tìm thấy {len(found_items)} mục tên '{specific_item_name}'. Xử lý mục đầu tiên.")
            items_to_process.append(found_items[0])
            print(f"✅ Đã tìm thấy mục cụ thể để xử lý.")
        except Exception as e:
            print(f"❌ Lỗi khi tìm mục cụ thể: {e}")
            return
    else:
        print("\nĐang liệt kê tất cả mục trong 'Được chia sẻ với tôi'. Vui lòng đợi...")
        page_token = None
        while True:
            try:
                response = drive_service.files().list(q="sharedWithMe and trashed = false",
                    fields="nextPageToken, files(id, name, mimeType)", pageSize=1000, pageToken=page_token).execute()
                items_to_process.extend(response.get('files', []))
                page_token = response.get('nextPageToken', None)
                if page_token is None: break
            except Exception as e:
                print(f"❌ Lỗi khi liệt kê tất cả mục được chia sẻ: {e}")
                return

    # Step 3: Filter the list based on scope and existence
    shared_files = [item for item in items_to_process if item['mimeType'] != 'application/vnd.google-apps.folder']
    shared_folders = [item for item in items_to_process if item['mimeType'] == 'application/vnd.google-apps.folder']

    if copy_scope == "Files Only": shared_folders = []
    elif copy_scope == "Folders Only": shared_files = []

    print(f"\nTìm thấy {len(shared_files)} file và {len(shared_folders)} thư mục cần xử lý.")
    existing_top_level_items = get_existing_items(dest_root_id) if skip_existing_items else {}
    files_to_copy = [f for f in shared_files if f['name'] not in existing_top_level_items]
    folders_to_copy = [f for f in shared_folders if f['name'] not in existing_top_level_items]
    print(f"Sẽ sao chép {len(files_to_copy)} file mới và {len(folders_to_copy)} thư mục mới đến đích.")

    # Step 4: Process the filtered lists
    if files_to_copy:
        print("\nĐang sao chép file cấp trên cùng...")
        def build_top_file_copy_request(batch, file_item):
            batch.add(drive_service.files().copy(fileId=file_item['id'], body={'name': file_item['name'], 'parents': [dest_root_id]}))
        execute_batch_in_chunks(files_to_copy, build_top_file_copy_request)

    newly_created_top_folders = {}
    if folders_to_copy:
        print("\nĐang tạo thư mục cấp trên cùng...")
        def top_folder_callback(request_id, response, exception):
             if not exception: newly_created_top_folders[request_id] = response
        def build_top_folder_create_request(batch, folder_item):
            metadata = {'name': folder_item['name'], 'mimeType': 'application/vnd.google-apps.folder', 'parents': [dest_root_id]}
            batch.add(drive_service.files().create(body=metadata, fields='id, name'), request_id=folder_item['id'], callback=top_folder_callback)
        execute_batch_in_chunks(folders_to_copy, build_top_folder_create_request)

    for folder in folders_to_copy:
        if folder['id'] in newly_created_top_folders:
            dest_folder_id = newly_created_top_folders[folder['id']]['id']
            copy_folder_contents(folder['id'], dest_folder_id, folder['name'])

    print("\n\n✅ Hoàn tất quá trình!")


# --- Run the main function ---
if __name__ == "__main__":
    main()
